[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/fast_track/22_web_scraping.ipynb)

# 📓 Notebook 22 (fast track) — Web Scraping & Open APIs

> **Module:** Web Scraping · **Estimated time:** ~65 minutes · **Difficulty:** Intermediate

> 🏎️ **You're on the fast track.** This is a **trimmed fast-track copy** of the canonical [`16_webscraping/`](../16_webscraping/) module — **NB 47** web-scraping fundamentals, **NB 48** scraping with Firecrawl, and **NB 49** the OpenAlex scholarly API. It keeps the essential arc — the acquisition ladder, BeautifulSoup on real HTML, polite scraping, and *"check for an API first"* — and drops the deep dives (multi-page crawling, schema-driven extraction, the full OpenAlex graph). Open the canonical module when you want that depth.

Most of the world's data has **no API**. It sits in HTML pages built for human eyes — product listings, catalogues, tables, articles. **Web scraping** is the craft of extracting that data anyway: fetch the page, parse the markup, pull out the pieces you want — *politely, legally, and without your pipeline breaking every time a designer moves a `<div>`.* But scraping is a **last resort**, and this notebook is as much about *when not to scrape* as how to.

> 🧭 **Mental model — the data-acquisition ladder.** Climb *down* it and stop at the first rung that works. **Is there an open API?** Use it — clean JSON, stable fields, sanctioned (§7, OpenAlex). **No API, but static HTML?** Parse it **politely** with `requests` + BeautifulSoup (§2–§6). **JavaScript-heavy or anti-bot?** Reach for a **managed scraper** (§8, Firecrawl). Every rung down costs you more fragility — so always try the rung above first.

> 📎 **Prerequisites:** fast-track **NB 9** (APIs & SQL — HTTP requests, status codes, JSON) and **NB 6** (pandas — the payoff of every scrape is a tidy `DataFrame`). Everything here runs **100% offline** on inline HTML fixtures and canned JSON.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Walk the **data-acquisition ladder** and decide *whether to scrape at all* — API > bulk/RSS > static HTML > managed scraper.
2. Read HTML as a **DOM tree** and parse it with **BeautifulSoup** — `find` / `find_all` and CSS `.select` — reading text, attributes, and links.
3. Turn an HTML **table into a pandas DataFrame**.
4. Scrape **politely**: an honest `User-Agent`, a `time.sleep` throttle, a response cache, and a `robots.txt` check with `urllib.robotparser`.
5. **Check for an open API first** — query, filter, and **flatten OpenAlex JSON** into pandas, null-safely.
6. Recognise where DIY scraping **breaks** (JavaScript, anti-bot) and when to reach for a **managed scraper** like Firecrawl.

## 1. The data-acquisition ladder — when to scrape

Scraping is a **last resort**, not a first move. Before you write a single selector, climb down this ladder and **stop at the first "yes":**

1. **Is there an open API or data export?** → Use it. Stabler, faster, sanctioned — and the top rung of this notebook (§7, OpenAlex).
2. **Is there a bulk dataset, an RSS feed, or a sitemap?** → Use that.
3. **Is the page static, server-rendered HTML?** → Parse it politely with `requests` + BeautifulSoup (§2–§6).
4. **Is it JavaScript-heavy or behind anti-bot defences?** → Reach for a headless browser or a **managed scraper** (§8, Firecrawl).

Once you *do* scrape, you are a **guest on someone else's server**. ⚠️ These are the rules you don't break:

| Rule | What it means in practice |
|---|---|
| **Respect `robots.txt`** | never fetch a `Disallow`-ed path; honour `Crawl-delay` (§6) |
| **Read the Terms of Service** | some sites *contractually* forbid scraping — **public ≠ free-to-take** |
| **Rate-limit yourself** | a human clicks every few seconds; don't fire 100 req/s — throttle (§6) |
| **Identify yourself** | send an honest `User-Agent` with a contact URL; never spoof one to evade a block |
| **No personal data** | names, emails, faces → **GDPR / CCPA** territory; no PII without a lawful basis |
| **Cache, don't re-fetch** | store what you pull so a re-run never hits the server twice (§6) |

> ⚖️ **Not legal advice.** Scraping law varies by country and is still evolving. Public data scraped politely is generally lower-risk; bypassing logins, ignoring the ToS, or collecting PII is higher-risk. When in doubt, use an official API.

> 🧠 **The four-step pipeline.** Every scraper — hand-rolled or managed — is the same shape: **FETCH** (HTTP GET) → **PARSE** (HTML → tree) → **EXTRACT** (pick the nodes) → **STORE** (rows / JSON). You already know *fetch* from fast-track **NB 9**; this notebook owns **parse + extract**.

> ⚠️ **`requests.get()` returns only the raw HTML the server sent — it does *not* run JavaScript.** If a page draws its content in the browser with JS, `requests` sees an empty shell. That limit is exactly where DIY scraping ends and §8 begins.

## Setup

One import block for the whole notebook. Everything runs **100% offline**: a real scraper starts with `requests.get(url)`, but to stay deterministic and network-free we ship a **tiny mock website as Python strings** and parse *that* — every technique is identical to the real thing; only the *source* of the HTML changes. `requests` is imported only so we can show the real call in comments.

In [1]:
import re, time, requests            # `requests` imported for reference only — we never hit the network
import pandas as pd
from bs4 import BeautifulSoup         # the parser; pip install beautifulsoup4  (imports as bs4)
from urllib import robotparser        # standard-library robots.txt reader (§6)
from urllib.parse import urljoin      # turn relative links into absolute ones

BASE = "https://books.meridian.test"                         # our fictional bookshop's origin
UA = "MeridianCourseBot/1.0 (+https://example.com/botinfo)"  # an honest, contactable identity
HEADERS = {"User-Agent": UA}

# The first digit of an HTTP status code IS its meaning — a scraper's traffic light:
for c, meaning in [(200, "OK        -> parse it"),      (301, "redirect  -> follow Location"),
                   (403, "forbidden -> blocked"),       (404, "not found -> gone"),
                   (429, "slow down -> back off, retry"), (503, "server err-> retry politely")]:
    print(f"  {c}  {meaning}")

# In production the fetch is one line (see NB 9 for timeouts, retries, raise_for_status):
#     html = requests.get(BASE + "/catalogue.html", headers=HEADERS, timeout=10).text
# Here the offline fetch() (defined in §2) returns that same HTML from an in-memory fixture.
print("\nReady ✅  (offline — parsing inline HTML fixtures, never the network)")

  200  OK        -> parse it
  301  redirect  -> follow Location
  403  forbidden -> blocked
  404  not found -> gone
  429  slow down -> back off, retry
  503  server err-> retry politely

Ready ✅  (offline — parsing inline HTML fixtures, never the network)


## 2. Fetch & parse — HTML is a tree

HTML is not flat text — it is a **tree of nested tags**, the **DOM** (Document Object Model). One node contains others:

```text
html
└─ body
   └─ ul#catalogue
      ├─ li.book  (data-genre="scifi")     ← a node: a tag …
      │  ├─ h3.title  →  "Dune"            ← … with child nodes,
      │  ├─ p.author  →  "Frank Herbert"
      │  └─ p.price   →  "£9.99"           ← … text …
      └─ li.book  (data-genre="fantasy")   ← … and attributes.
```

Three things live on every node, and scraping is just reading them: the **tag name** (`tag.name`), its **attributes** (`tag["href"]`, `tag.get("data-genre")`), and its **text** (`tag.get_text(strip=True)`). Let's build our offline mock bookshop and parse its catalogue page.

In [2]:
# ── Our offline mock website: "Meridian Books" ───────────────────────────────
# Each value is exactly what requests.get(BASE + path).text would return for that
# page. Parsing these strings is identical to parsing a live response.

CATALOGUE_HTML = '''
<!DOCTYPE html>
<html lang="en">
<head><title>Meridian Books - Catalogue</title></head>
<body>
  <header><a id="logo" href="/index.html">Meridian Books</a></header>
  <main>
    <h1>Catalogue</h1>
    <ul id="catalogue">
      <li class="book" data-genre="scifi">
        <h3 class="title"><a class="book-link" href="/book/dune.html">Dune</a></h3>
        <p class="author">Frank Herbert</p>
        <p class="price">&pound;9.99</p>
        <p class="rating" data-stars="5">*****</p>
        <span class="availability in-stock">In stock (14)</span>
      </li>
      <li class="book" data-genre="fantasy">
        <h3 class="title"><a class="book-link" href="/book/the-hobbit.html">The Hobbit</a></h3>
        <p class="author">J. R. R. Tolkien</p>
        <p class="price">&pound;7.50</p>
        <p class="rating" data-stars="5">*****</p>
        <span class="availability in-stock">In stock (3)</span>
      </li>
      <li class="book" data-genre="nonfiction">
        <h3 class="title"><a class="book-link" href="/book/sapiens.html">Sapiens</a></h3>
        <p class="author">Yuval Noah Harari</p>
        <p class="price">&pound;12.00</p>
        <p class="rating" data-stars="4">****</p>
        <span class="availability out-of-stock">Out of stock</span>
      </li>
      <li class="book" data-genre="scifi">
        <h3 class="title"><a class="book-link" href="/book/neuromancer.html">Neuromancer</a></h3>
        <p class="author">William Gibson</p>
        <p class="price">&pound;8.25</p>
        <p class="rating" data-stars="4">****</p>
        <span class="availability in-stock">In stock (7)</span>
      </li>
      <li class="book" data-genre="scifi">
        <h3 class="title"><a class="book-link" href="/book/foundation.html">Foundation</a></h3>
        <p class="author">Isaac Asimov</p>
        <p class="price">&pound;6.99</p>
        <p class="rating" data-stars="4">****</p>
        <span class="availability out-of-stock">Out of stock</span>
      </li>
    </ul>
  </main>
  <footer>(c) 2026 Meridian Books &middot; <a href="/about.html">About</a></footer>
</body>
</html>
'''

# A book DETAIL page (used in Practice Exercise 2) — note the <table class="meta">.
DUNE_HTML = '''
<!DOCTYPE html>
<html lang="en">
<head><title>Dune - Meridian Books</title></head>
<body><main>
  <h1 class="title">Dune</h1>
  <p class="author">by Frank Herbert</p>
  <table class="meta">
    <tr><th>First published</th><td>1965</td></tr>
    <tr><th>Pages</th><td>412</td></tr>
    <tr><th>ISBN</th><td>978-0-441-17271-9</td></tr>
    <tr><th>Genre</th><td>Science fiction</td></tr>
  </table>
</main></body>
</html>
'''

# A standalone HTML table for §5 (this month's bestsellers).
BESTSELLERS_HTML = '''
<table id="bestsellers">
  <thead><tr><th>Rank</th><th>Title</th><th>Author</th><th>Copies sold</th></tr></thead>
  <tbody>
    <tr><td>1</td><td>Dune</td><td>Frank Herbert</td><td>12000</td></tr>
    <tr><td>2</td><td>The Hobbit</td><td>J. R. R. Tolkien</td><td>9800</td></tr>
    <tr><td>3</td><td>Sapiens</td><td>Yuval Noah Harari</td><td>8700</td></tr>
    <tr><td>4</td><td>Neuromancer</td><td>William Gibson</td><td>5400</td></tr>
  </tbody>
</table>
'''

# The site's posted house rules for bots (used in §6).
ROBOTS = '''
User-agent: *
Disallow: /cart/
Disallow: /checkout/
Disallow: /account/
Crawl-delay: 1
Allow: /

User-agent: GreedyBot
Disallow: /
'''

# The whole "site": a path -> HTML map. fetch() is our offline requests.get.
PAGES = {"/catalogue.html": CATALOGUE_HTML, "/book/dune.html": DUNE_HTML}

def fetch(path):
    '''Offline stand-in for requests.get(BASE + path, headers=HEADERS, timeout=10).text'''
    if path not in PAGES:
        raise KeyError("404 Not Found: " + path)   # a real GET would return status 404
    return PAGES[path]

# PARSE — turn the raw HTML string into a navigable tree. "html.parser" is built into
# Python (no lxml needed); pass "lxml" instead only if you have installed it.
soup = BeautifulSoup(fetch("/catalogue.html"), "html.parser")

print("Page <title>:", soup.title.get_text(strip=True))
print("The <h1>    :", soup.h1.get_text(strip=True))
first = soup.find("li", class_="book")          # first <li class="book"> node
print("First book  :", first.find("h3", class_="title").get_text(strip=True),
      "| genre attr:", first["data-genre"], "| parent:", first.parent.name)

Page <title>: Meridian Books - Catalogue
The <h1>    : Catalogue
First book  : Dune | genre attr: scifi | parent: ul


## 3. `find` / `find_all` — text, attributes, and links

Two methods do most of the work. **`find`** returns the **first** matching node (or `None`); **`find_all`** returns a **list** of every match. Filter by tag name, by `class_=`, by `id=`, or by any attribute.

| Goal | Code |
|---|---|
| First `<h1>` | `soup.find("h1")` |
| First `<li class="book">` | `soup.find("li", class_="book")` |
| **All** `<li class="book">` | `soup.find_all("li", class_="book")` |
| By id | `soup.find(id="catalogue")` |
| The node's text | `node.get_text(strip=True)` |
| An attribute value | `node["href"]` or `node.get("href")` |

> 🔬 **`["href"]` vs `.get("href")`.** `node["href"]` raises `KeyError` if the attribute is missing; `node.get("href")` returns `None` instead — the same choice as with dicts. A **link** is just the `href` attribute of an `<a>` tag; `urljoin(BASE, href)` turns a relative link into an absolute one.

In [3]:
# find_all -> a list of every book node; loop it and read text, attributes, and links.
book_nodes = soup.find_all("li", class_="book")
print(len(book_nodes), "books on the page:\n")
for li in book_nodes:
    title  = li.find("h3", class_="title").get_text(strip=True)
    author = li.find("p", class_="author").get_text(strip=True)
    price  = li.find("p", class_="price").get_text(strip=True)
    genre  = li["data-genre"]                              # attribute access
    href   = li.find("a", class_="book-link").get("href")  # .get -> None if missing
    link   = urljoin(BASE, href)                           # relative -> absolute URL
    print(f"  {title:22} {price:>7}  [{genre:10}]  {author:18}  -> {link}")

5 books on the page:

  Dune                     £9.99  [scifi     ]  Frank Herbert       -> https://books.meridian.test/book/dune.html
  The Hobbit               £7.50  [fantasy   ]  J. R. R. Tolkien    -> https://books.meridian.test/book/the-hobbit.html
  Sapiens                 £12.00  [nonfiction]  Yuval Noah Harari   -> https://books.meridian.test/book/sapiens.html
  Neuromancer              £8.25  [scifi     ]  William Gibson      -> https://books.meridian.test/book/neuromancer.html
  Foundation               £6.99  [scifi     ]  Isaac Asimov        -> https://books.meridian.test/book/foundation.html


---

### ✋ Quick exercise (~2 min) — Read one book off the page

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using the existing `soup` from §2 (no re-fetching), grab the **first** book node with `soup.find("li", class_="book")`, then pull its **title** text and its **rating stars** (the `data-stars` attribute on the `<p class="rating">`). Print them.

In [4]:
# ✍️ Your turn 👇
# Use the existing `soup` from §2 — no re-fetching.
first_book = soup.find("li", class_="book")
title = ...      # the book's title text (inside <h3 class="title">)
stars = ...      # the data-stars attribute on <p class="rating">  (comes back a string)
# print(title, "-", stars, "stars")

<details>
<summary>✅ <b>Solution</b></summary>

```python
first_book = soup.find("li", class_="book")
title = first_book.find("h3", class_="title").get_text(strip=True)
stars = first_book.find("p", class_="rating")["data-stars"]
print(title, "-", stars, "stars")
```

`find` returns the first matching node; `.get_text(strip=True)` reads its text, and `[...]` reads an attribute. Attributes always come back as **strings** — wrap in `int(stars)` if you need a number.
</details>

## 4. CSS selectors → a tidy DataFrame

`find`/`find_all` work, but **CSS selectors** are often shorter and more expressive — the very same selectors you'd use in a browser's dev tools. Two methods: **`soup.select("…")`** returns a **list** (like `find_all`); **`soup.select_one("…")`** returns the **first** match (like `find`).

| Selector | Matches |
|---|---|
| `li.book` | `<li>` that also has class `book` |
| `.author` | any element with `class="author"` |
| `#catalogue li.book` | `li.book` anywhere inside `#catalogue` (descendant) |
| `a.book-link` | `<a>` with class `book-link` |
| `[data-genre="scifi"]` | any element whose `data-genre` equals `scifi` |

Let's use selectors to extract **every** book into a tidy list of dicts — then a DataFrame, the whole point of a scrape.

In [5]:
# A reusable extractor: one <li class="book"> node -> a clean, typed dict of fields.
def parse_books(page_soup):
    rows = []
    for li in page_soup.select("li.book"):                       # every <li class="book">
        avail = li.select_one(".availability").get_text(strip=True)
        rows.append({
            "title":    li.select_one(".title").get_text(strip=True),
            "author":   li.select_one(".author").get_text(strip=True),
            "price":    float(re.sub("[^0-9.]", "", li.select_one(".price").get_text())),
            "stars":    int(li.select_one(".rating")["data-stars"]),   # attribute -> int
            "genre":    li["data-genre"],
            "in_stock": "In stock" in avail,
            "url":      li.select_one("a.book-link")["href"],
        })
    return rows

catalogue_df = pd.DataFrame(parse_books(soup))
print(catalogue_df.to_string(index=False))
print("\nscifi titles via attribute selector:",
      [a.get_text(strip=True) for a in soup.select('li[data-genre="scifi"] .title')])
print("Cheapest in-stock book:",
      catalogue_df[catalogue_df["in_stock"]].sort_values("price").iloc[0]["title"])

      title            author  price  stars      genre  in_stock                    url
       Dune     Frank Herbert   9.99      5      scifi      True        /book/dune.html
 The Hobbit  J. R. R. Tolkien   7.50      5    fantasy      True  /book/the-hobbit.html
    Sapiens Yuval Noah Harari  12.00      4 nonfiction     False     /book/sapiens.html
Neuromancer    William Gibson   8.25      4      scifi      True /book/neuromancer.html
 Foundation      Isaac Asimov   6.99      4      scifi     False  /book/foundation.html

scifi titles via attribute selector: ['Dune', 'Neuromancer', 'Foundation']
Cheapest in-stock book: The Hobbit


## 5. An HTML table → pandas

An HTML table is its own little tree: `<table>` → `<thead>`/`<tbody>` → `<tr>` (rows) → `<th>` (headers) / `<td>` (cells). Walk it once and you have a DataFrame.

> 💡 **In production you'd reach for `pd.read_html(html)`** — it parses *every* `<table>` on a page into a list of DataFrames. But it needs a parser backend (`lxml` / `html5lib`) which this offline environment doesn't have, so below we do it **by hand with BeautifulSoup** — no extra dependency, and it shows you exactly what `read_html` does under the hood.

In [6]:
# Parse an HTML <table> into a DataFrame by hand (what pd.read_html does for you).
tbl = BeautifulSoup(BESTSELLERS_HTML, "html.parser").select_one("table#bestsellers")

col_names = [th.get_text(strip=True) for th in tbl.select("thead th")]
rows = [[td.get_text(strip=True) for td in tr.select("td")] for tr in tbl.select("tbody tr")]

bestsellers_df = pd.DataFrame(rows, columns=col_names)
bestsellers_df["Copies sold"] = bestsellers_df["Copies sold"].astype(int)  # cells arrive as str
print(bestsellers_df.to_string(index=False))
print("\nTotal copies sold:", int(bestsellers_df["Copies sold"].sum()))
# In production, given an lxml/html5lib backend, this is one line:
#     bestsellers_df = pd.read_html(BESTSELLERS_HTML)[0]

Rank       Title            Author  Copies sold
   1        Dune     Frank Herbert        12000
   2  The Hobbit  J. R. R. Tolkien         9800
   3     Sapiens Yuval Noah Harari         8700
   4 Neuromancer    William Gibson         5400

Total copies sold: 35900


## 6. Polite scraping — the rules you don't break

A well-behaved scraper does four things, **every** request. Two you met in NB 9 (timeouts, retries); the scraping-specific ones are **throttling** and **caching**, plus the **`robots.txt`** check.

1. ⚠️ **Identify** — send an honest `User-Agent` with a contact URL (our `HEADERS`). Never spoof one to evade a block.
2. ⚠️ **Throttle** — `time.sleep(delay)` between requests; honour any `Crawl-delay`. Never fire unbounded parallel hits.
3. ⚠️ **Cache** — store every response so a re-run never re-fetches the same URL. Kind to the server, fast for you.
4. ⚠️ **Obey `robots.txt`** — check *before* you knock. Python's standard library reads and enforces it for you.

First, the robots check with `urllib.robotparser` — point it at the site's `robots.txt` and ask *"may I fetch this?"*

In [7]:
# urllib.robotparser is standard library: it parses robots.txt and answers "may I?"
rp = robotparser.RobotFileParser()
rp.parse(ROBOTS.splitlines())          # in production: rp.set_url(BASE + "/robots.txt"); rp.read()

for p in ["/catalogue.html", "/cart/checkout", "/account/settings", "/book/dune.html"]:
    print(f"  {'allowed' if rp.can_fetch(UA, BASE + p) else 'BLOCKED'}  {p}")

print("\nCrawl-delay requested:", rp.crawl_delay(UA), "second(s) -> sleep at least this long")
print("GreedyBot may fetch '/':", rp.can_fetch("GreedyBot", BASE + "/"), "(banned outright)")

  allowed  /catalogue.html
  BLOCKED  /cart/checkout
  BLOCKED  /account/settings
  allowed  /book/dune.html

Crawl-delay requested: 1 second(s) -> sleep at least this long
GreedyBot may fetch '/': False (banned outright)


In [8]:
# A polite fetch = User-Agent + throttle + cache + robots-awareness, all offline here.
_cache, _last = {}, [0.0]
DELAY = 0.1          # short demo delay; honour the site's Crawl-delay (1s here) in production

def polite_get(path):
    if not rp.can_fetch(UA, BASE + path):
        print(f"  robots-skip  {path}"); return None          # OBEY: never fetch a disallowed path
    if path in _cache:
        print(f"  cache-hit    {path}"); return _cache[path]  # CACHE: never re-fetch
    wait = DELAY - (time.monotonic() - _last[0])
    if wait > 0:
        time.sleep(wait)                                      # THROTTLE: respect the gap
    html = fetch(path)   # prod: requests.get(BASE + path, headers=HEADERS, timeout=10).text
    _last[0] = time.monotonic(); _cache[path] = html
    print(f"  FETCH        {path}")
    return html

for p in ["/catalogue.html", "/catalogue.html", "/account/settings"]:
    polite_get(p)       # FETCH, then a cache-hit, then a robots-skip

  FETCH        /catalogue.html
  cache-hit    /catalogue.html
  robots-skip  /account/settings


---

### ✋ Quick exercise (~2 min) — Ask robots first

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A polite crawler checks the rules *before* it knocks. Reusing `rp` and `UA` from §6, loop over the paths below and print, for each, whether a bot is allowed to fetch it. How many of the four are allowed?

```python
paths = ["/catalogue.html", "/cart/", "/book/dune.html", "/account/orders"]
```

In [9]:
# ✍️ Your turn 👇  (reuse `rp` and `UA` from §6)
paths = ["/catalogue.html", "/cart/", "/book/dune.html", "/account/orders"]
allowed = ...      # count how many of `paths` rp.can_fetch(UA, BASE + path) allows
# for p in paths:
#     print(p, rp.can_fetch(UA, BASE + p))

<details>
<summary>✅ <b>Solution</b></summary>

```python
paths = ["/catalogue.html", "/cart/", "/book/dune.html", "/account/orders"]
allowed = 0
for p in paths:
    ok = rp.can_fetch(UA, BASE + p)
    print(f"  {'allowed' if ok else 'BLOCKED'}  {p}")
    allowed += ok
print(allowed, "of", len(paths), "allowed")
```

`/cart/` and `/account/orders` sit under `Disallow` rules, so `can_fetch` returns `False`; the catalogue and book paths are allowed → **2 of 4**. Checking `can_fetch` *before* fetching means a disallowed URL never even enters your queue. (`allowed += ok` works because `True`/`False` count as `1`/`0`.)
</details>

## 7. Check for an open API first — OpenAlex

Everything above was the **third rung** of the ladder: no API, so parse the HTML politely. Now the **top rung**. Imagine the task *"collect the most-cited graph-learning papers with their authors and venues."* You *could* scrape Google Scholar — fragile, slow, against its ToS. Or you query **[OpenAlex](https://openalex.org)**: a free, fully-open index of **~250 million** scholarly works, no API key required, returning clean JSON with **stable field names**.

| | 🕸️ Scrape a journal site | 🔌 Query OpenAlex |
|---|---|---|
| **Data shape** | HTML you must parse; layout changes break you | structured JSON, stable field names |
| **Coverage** | one publisher at a time | ~250M works across all publishers |
| **Blocking** | CAPTCHAs, IP bans, ToS violations | documented limits, *invites* automated use |
| **Auth / cost** | often a login | **no key**, free, CC0 data |

> 🧠 **The rule: check for an API first.** When a clean API exists, scraping is the *wrong tool*. A real call is `requests.get("https://api.openalex.org/works?filter=...&mailto=you@example.com")`. To stay offline we parse a small **canned snapshot** of Works using the *real* OpenAlex field names — pinning a snapshot is also good practice for reproducible analysis.

In [10]:
# A tiny pinned snapshot of OpenAlex "Works" (real field names; mock values for teaching).
# A live GET https://api.openalex.org/works?... returns exactly this nested shape.
CANNED_WORKS = [
    {"id": "https://openalex.org/W4001",
     "display_name": "Semi-Supervised Classification with Graph Convolutional Networks",
     "publication_year": 2019, "cited_by_count": 21000,
     "authorships": [{"author": {"display_name": "Thomas N. Kipf"},
                      "institutions": [{"display_name": "University of Amsterdam"}]}],
     "primary_location": {"source": {"display_name": "ICLR", "type": "conference"}},
     "open_access": {"is_oa": True}},
    {"id": "https://openalex.org/W4002",
     "display_name": "Inductive Representation Learning on Large Graphs",
     "publication_year": 2019, "cited_by_count": 15000,
     "authorships": [{"author": {"display_name": "William L. Hamilton"},
                      "institutions": [{"display_name": "Stanford University"}]}],
     "primary_location": {"source": {"display_name": "NeurIPS", "type": "conference"}},
     "open_access": {"is_oa": True}},
    {"id": "https://openalex.org/W4003",
     "display_name": "Graph Attention Networks",
     "publication_year": 2020, "cited_by_count": 18000,
     "authorships": [{"author": {"display_name": "Petar Velickovic"},
                      "institutions": [{"display_name": "University of Cambridge"}]}],
     "primary_location": {"source": {"display_name": "ICLR", "type": "conference"}},
     "open_access": {"is_oa": True}},
    {"id": "https://openalex.org/W4004",
     "display_name": "How Powerful are Graph Neural Networks?",
     "publication_year": 2020, "cited_by_count": 9000,
     "authorships": [{"author": {"display_name": "Keyulu Xu"},
                      "institutions": [{"display_name": "MIT"}]}],
     "primary_location": {"source": {"display_name": "ICLR", "type": "conference"}},
     "open_access": {"is_oa": True}},
    {"id": "https://openalex.org/W4005",
     "display_name": "A Comprehensive Survey on Graph Neural Networks",
     "publication_year": 2021, "cited_by_count": 7000,
     "authorships": [{"author": {"display_name": "Zonghan Wu"},
                      "institutions": [{"display_name": "University of Technology Sydney"}]}],
     "primary_location": {"source": {"display_name": "IEEE TNNLS", "type": "journal"}},
     "open_access": {"is_oa": False}},
    {"id": "https://openalex.org/W4006",
     "display_name": "Open Graph Benchmark: Datasets for ML on Graphs",
     "publication_year": 2021, "cited_by_count": 3000,
     "authorships": [{"author": {"display_name": "Weihua Hu"},
                      "institutions": [{"display_name": "Stanford University"}]}],
     "primary_location": None,          # a preprint with no indexed venue -> null (real data is messy!)
     "open_access": {"is_oa": True}},
]

# Reading a Work is just dictionary access — no HTML, no CSS selectors, stable fields:
w0 = CANNED_WORKS[0]
print(w0["display_name"])
print(f"  year {w0['publication_year']}  |  {w0['cited_by_count']:,} citations")
print("  first author:", w0["authorships"][0]["author"]["display_name"])
print("  venue       :", w0["primary_location"]["source"]["display_name"])

Semi-Supervised Classification with Graph Convolutional Networks
  year 2019  |  21,000 citations
  first author: Thomas N. Kipf
  venue       : ICLR


> 🔬 **The anatomy of a Work.** Every result carries the same nested shape: `id`, `display_name`, `publication_year`, `cited_by_count`, an `authorships[]` list (`author.display_name`, `institutions[]`), and `primary_location.source` (the venue — **which can be `null`**). On the live API you narrow results *server-side* with **`filter=`** (exact constraints, e.g. `filter=publication_year:2019`) and **`search=`** (fuzzy full-text) — you ask a precise question and get a precise answer, never downloading 250M records. We mimic a `filter` in Python, then **flatten** the nested JSON into a tidy DataFrame — guarding every nullable field.

In [11]:
# 1) A server-side filter (filter=publication_year:2019) is a one-liner here:
works_2019 = [w["display_name"] for w in CANNED_WORKS if w["publication_year"] == 2019]
print("Works from 2019:", works_2019, "\n")

# 2) Flatten nested JSON -> one flat row per Work, guarding every nullable field.
def flatten_work(w):
    authors = w.get("authorships") or []
    first_author = authors[0]["author"]["display_name"] if authors else None
    # primary_location AND its source can each be null -> chain with "or {}"
    source = (w.get("primary_location") or {}).get("source") or {}
    return {
        "openalex_id":    w["id"].rsplit("/", 1)[-1],        # W4001
        "title":          w["display_name"],
        "year":           w["publication_year"],
        "cited_by_count": w["cited_by_count"],
        "first_author":   first_author,
        "venue":          source.get("display_name"),        # None if the venue is unknown
        "is_oa":          (w.get("open_access") or {}).get("is_oa"),
    }

openalex_df = pd.DataFrame([flatten_work(w) for w in CANNED_WORKS])
print(openalex_df.to_string(index=False))
print("\nTotal citations by year:")
print(openalex_df.groupby("year")["cited_by_count"].sum().sort_index().to_string())

Works from 2019: ['Semi-Supervised Classification with Graph Convolutional Networks', 'Inductive Representation Learning on Large Graphs'] 

openalex_id                                                            title  year  cited_by_count        first_author      venue  is_oa
      W4001 Semi-Supervised Classification with Graph Convolutional Networks  2019           21000      Thomas N. Kipf       ICLR   True
      W4002                Inductive Representation Learning on Large Graphs  2019           15000 William L. Hamilton    NeurIPS   True
      W4003                                         Graph Attention Networks  2020           18000    Petar Velickovic       ICLR   True
      W4004                          How Powerful are Graph Neural Networks?  2020            9000           Keyulu Xu       ICLR   True
      W4005                  A Comprehensive Survey on Graph Neural Networks  2021            7000          Zonghan Wu IEEE TNNLS  False
      W4006                  Open Gra

> 🧠 **The `or {}` trick.** `(w.get("primary_location") or {}).get("source") or {}` walks a nested path that might dead-end in `None` at any step, without raising — the pandas-boundary version of *"never trust the shape of external data."* One Work here (`W4006`, a preprint) has `primary_location: None`, so its `venue` lands as `None` instead of crashing the flatten.

---

### ✋ Quick exercise (~2 min) — A null-safe venue count

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Reading OpenAlex is just dict access — but nested fields can be `null`. Using `CANNED_WORKS`, count how many works have an **unknown venue**. The venue lives at `work["primary_location"]["source"]["display_name"]`, but `primary_location` (and `source`) can be `None` — use the `or {}` trick so you never raise.

In [12]:
# ✍️ Your turn 👇  (reuse CANNED_WORKS from §7)
def venue_of(work):
    source = (work.get("primary_location") or {}).get("source") or {}
    return source.get("display_name")     # None if the venue is unknown

unknown = ...      # count works where venue_of(work) is None
# print(unknown, "of", len(CANNED_WORKS), "works have an unknown venue")

<details>
<summary>✅ <b>Solution</b></summary>

```python
def venue_of(work):
    source = (work.get("primary_location") or {}).get("source") or {}
    return source.get("display_name")

unknown = sum(venue_of(w) is None for w in CANNED_WORKS)
print(unknown, "of", len(CANNED_WORKS), "works have an unknown venue")
```

Only `W4006` (a preprint with `primary_location: None`) has no venue → **1 of 6**. The `or {}` chain means a `None` at *any* level short-circuits to an empty dict, so `.get("display_name")` returns `None` instead of raising `AttributeError`. Guard external data at the boundary — then a missing field is a value you can count, not a crash.
</details>

## 8. When to reach for a managed scraper — Firecrawl

`requests` + BeautifulSoup is perfect for **static, server-rendered HTML** — like every page above. The modern web often isn't. When DIY breaks, you've hit the **bottom rung** of the ladder:

| Problem | Why `requests` + `bs4` struggles | The usual fix |
|---|---|---|
| **JavaScript rendering** | `requests` gets the empty shell; JS draws the content in a browser | a **headless browser** (Playwright / Selenium) |
| **Anti-bot / Cloudflare** | challenges, fingerprinting, IP blocks | rotating proxies, stealth browsers, or a **managed service** |
| **Layout drift** | selectors break on every redesign → `None`, then `AttributeError` | resilient selectors, monitoring, or LLM extraction |
| **"Just give me clean text for my LLM"** | HTML is full of nav, ads, boilerplate | a service that returns clean **markdown** |

> 🧭 **The conceptual escape hatch (no install here).** A **managed scraper** like **[Firecrawl](https://firecrawl.dev)** hands the whole mess — a headless browser, proxies, anti-bot, boilerplate stripping — to an API. You call `app.scrape(url, formats=["markdown"])` and get clean, **LLM-ready markdown** plus metadata; or pass a **schema** and get **typed JSON** straight from the page, *no selectors, resilient to layout changes*. The full walkthrough (`scrape` / `crawl` / `map`, schema extraction, cost & rate limits) is the canonical **NB 48**:
>
> ```python
> # pip install firecrawl-py ; export FIRECRAWL_API_KEY=fc-...
> from firecrawl import Firecrawl
> app = Firecrawl()                                   # reads FIRECRAWL_API_KEY from the environment
> doc = app.scrape("https://example.com", formats=["markdown"])
> print(doc.markdown)                                 # clean, LLM-ready text — no <script>, no nav
> ```

> ⚠️ **It's not free money.** Every `scrape` is a billed credit and a network round-trip; a `crawl` can be hundreds. The politeness habits from §6 still apply — **throttle, cache, and set a `limit`** — even though the *anti-bot* part is now the service's problem, not yours. Climb the ladder from the top: API first, then polite DIY, and only then a managed scraper.

## 🧪 Practice exercises

Four exercises, warm-up to a bug-hunt. They reuse the fixtures and helpers built above (`soup`, `fetch`, `catalogue_df`, `openalex_df`, `DUNE_HTML`). Try each before opening the solution.

### Exercise 1 — ⭐ Titles and prices

From the catalogue `soup`, build a list of `(title, price_text)` tuples — one per book — using either `find_all` or `.select`. Keep the price as *text* (with its `£`). Print them.

In [13]:
# Your code here 👇
pairs = ...
# for t, p in pairs:
#     print(f"{t:22} {p}")

<details>
<summary>💡 <b>Solution</b></summary>

```python
pairs = [(li.select_one(".title").get_text(strip=True),
          li.select_one(".price").get_text(strip=True))
         for li in soup.select("li.book")]
for t, p in pairs:
    print(f"{t:22} {p}")
```

One comprehension over `soup.select("li.book")` pulls both fields per node. Keeping the price as *text* preserves the currency symbol — parse to a `float` only when you need to compute.
</details>

### Exercise 2 — ⭐⭐ Parse a detail page's table

Fetch the Dune detail page (`fetch("/book/dune.html")`), parse it, and extract its `<table class="meta">` into a **dict** like `{"First published": "1965", "Pages": "412", ...}`. Each row is a `<tr>` holding a `<th>` (key) and a `<td>` (value).

In [14]:
# Your code here 👇
detail = BeautifulSoup(fetch("/book/dune.html"), "html.parser")
meta = ...
# print(meta)

<details>
<summary>💡 <b>Solution</b></summary>

```python
detail = BeautifulSoup(fetch("/book/dune.html"), "html.parser")
meta = {}
for tr in detail.select("table.meta tr"):
    meta[tr.find("th").get_text(strip=True)] = tr.find("td").get_text(strip=True)
print(meta)
```

Each `<tr>` pairs one `<th>` (label) with one `<td>` (value) — reading them row by row rebuilds the record as a dict. That dict-per-item shape is exactly what you'd write to a database row.
</details>

### Exercise 3 — ⭐⭐ The API payoff — citations per venue

Using `openalex_df` from §7, compute the **total `cited_by_count` per `venue`**, sorted highest first. Replace the null venue (the preprint) with a clear label like `"(unknown / preprint)"` instead of `NaN`. (This is the pandas payoff of an API pull — NB 6.)

In [15]:
# Your code here 👇
by_venue = ...
# print(by_venue)

<details>
<summary>💡 <b>Solution</b></summary>

```python
by_venue = (openalex_df.assign(venue=openalex_df["venue"].fillna("(unknown / preprint)"))
            .groupby("venue")["cited_by_count"].sum()
            .sort_values(ascending=False))
print(by_venue.to_string())
```

`fillna` turns the null venue into a labelled bucket so it survives the `groupby` instead of being dropped; `groupby(...).sum()` then totals citations per venue. Because the data arrived as clean, typed JSON, this is a one-liner — no parsing, no guessing.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The helper below is meant to read each book's **discount badge**, but it **crashes**. Run it, read the traceback, work out the root cause, then write a fix that returns a sensible default when there is no discount.

In [16]:
# 🐞 Buggy on purpose — run it, see the error, then write your fix below.
def get_discount(book_li):
    # our book cards have NO <span class="discount">, so .find(...) returns None
    return book_li.find("span", class_="discount").text

first_book = soup.find("li", class_="book")
print(get_discount(first_book))     # AttributeError: 'NoneType' object has no attribute 'text'

AttributeError: 'NoneType' object has no attribute 'text'

<details>
<summary>💡 <b>Solution</b></summary>

**Root cause.** `book_li.find("span", class_="discount")` finds nothing (no book has a discount badge), so it returns `None`. Calling `.text` on `None` raises `AttributeError: 'NoneType' object has no attribute 'text'`. This is *the* classic scraping bug — and exactly how a silent layout change surfaces at runtime.

```python
def get_discount(book_li, default="no discount"):
    badge = book_li.find("span", class_="discount")    # may be None
    return badge.get_text(strip=True) if badge else default

first_book = soup.find("li", class_="book")
print(get_discount(first_book))                         # -> "no discount"
```

**Lesson.** Never chain `.text` / `.get_text()` straight onto a `find` / `select_one` result you haven't checked. Capture the node, test it for `None`, *then* read it — the discipline that keeps a scraper alive through a redesign.
</details>

## 🧠 Stretch exercises

Two harder, open-ended problems. Each solution ends with the *why*.

### Stretch exercise C — ⭐⭐⭐ A resilient row extractor

Write `safe_parse_book(li)` that returns a dict with `title`, `price`, and `stars`, but **never crashes** on a missing field — use `None` (or a default) for anything absent. Test it on a normal book node *and* on a broken one: `BeautifulSoup("<li class='book'></li>", "html.parser").li`.

In [17]:
# Your code here 👇
def safe_parse_book(li):
    ...
# print(safe_parse_book(soup.find("li", class_="book")))
# print(safe_parse_book(BeautifulSoup("<li class='book'></li>", "html.parser").li))

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def safe_parse_book(li, default=None):
    def text(sel):
        node = li.select_one(sel)
        return node.get_text(strip=True) if node else default
    rating = li.select_one(".rating")
    return {
        "title": text(".title"),
        "price": text(".price"),
        "stars": int(rating["data-stars"]) if rating and rating.has_attr("data-stars") else default,
    }

print(safe_parse_book(soup.find("li", class_="book")))
print(safe_parse_book(BeautifulSoup("<li class='book'></li>", "html.parser").li))
```

**Reasoning.** A tiny `text()` helper centralises the `None`-guard so every field is defended the same way, and the broken node returns `{"title": None, "price": None, "stars": None}` instead of raising. That is the difference between a scraper that *survives* a layout change and one that dies on the first redesign — and because the failure mode is an all-`None` row rather than a crash, you can **detect** it (alert on empty rows) instead of getting paged at 3 a.m. Defensive parsing is what turns a one-off script into a pipeline you can leave running.
</details>

### Stretch exercise D — ⭐⭐⭐ A credit-saving cache, counted

Politeness rule #3 from §6 was *cache, don't re-fetch*. Write a thin `CountingCache` wrapper around `fetch` that memoises responses by path **and** counts network **fetches** vs **cache hits**. Fetch `/catalogue.html` three times and `/book/dune.html` once, then report `2 fetches, 2 cache hits`.

In [18]:
# Your code here 👇
# Wrap `fetch` in a class with a dict cache and two counters (fetches, hits).
# Then call it: /catalogue.html x3, /book/dune.html x1  ->  "2 fetches, 2 cache hits".

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class CountingCache:
    def __init__(self, fetch_fn):
        self.fetch_fn, self.cache = fetch_fn, {}
        self.fetches = self.hits = 0
    def get(self, path):
        if path in self.cache:
            self.hits += 1
            return self.cache[path]
        self.fetches += 1
        self.cache[path] = self.fetch_fn(path)
        return self.cache[path]

c = CountingCache(fetch)
for p in ["/catalogue.html", "/catalogue.html", "/catalogue.html", "/book/dune.html"]:
    c.get(p)
print(f"{c.fetches} fetches, {c.hits} cache hits")
```

**Reasoning.** Two *distinct* URLs → **2 fetches**; the two repeat requests for the catalogue → **2 cache hits**. A cache is the single biggest politeness win a scraper has: it turns re-runs — the thing you do constantly while developing — into **zero** extra server load and near-instant responses, and it applies just as much to a *paid* managed API (§8), where every avoided call is a saved credit. The same "never fetch the same thing twice" rule spans all three rungs of the ladder: cache your scrapes, cache your API calls, cache your managed-scraper credits.
</details>

## 🧠 Key takeaways

1. **Climb the acquisition ladder from the top.** Open **API** > bulk/RSS > polite **static-HTML** scrape > **managed scraper**. Every rung down costs more fragility — always try the rung above first.
2. **A web page is a DOM tree.** Scraping is **fetch → parse → extract → store**; this notebook owns *parse + extract*. Parse with **`BeautifulSoup(html, "html.parser")`** — the standard-library parser, no `lxml` needed.
3. **Two ways to reach a node:** `find` / `find_all` and CSS `.select` / `.select_one`. Read text with `get_text(strip=True)`, attributes and links with `node["attr"]` / `node.get("attr")`, and land the result in a **pandas DataFrame** (tables too, by walking `thead`/`tbody`).
4. **Be polite, every request:** an honest `User-Agent`, a `time.sleep` throttle, a response **cache**, and a `urllib.robotparser` `can_fetch` check *before* you knock.
5. **Check for an API first.** OpenAlex returns clean, stable JSON — reading it is dict access, not selector-guessing. **Flatten null-safely** with the `(x or {}).get(...)` trick before it hits pandas.
6. **Selectors are brittle.** Guard every lookup for `None` (the classic `None.text` `AttributeError`), prefer stable hooks, and alert on zero rows — a redesign breaks scrapers *silently*.
7. **DIY breaks on JavaScript and anti-bot pages** → a headless browser or a managed API (Firecrawl) that returns clean markdown or typed JSON — the canonical **NB 48**.

## ✅ Self-assessment

- [ ] Walk the data-acquisition ladder and say when to use an API vs. scrape vs. a managed service
- [ ] Name three rules you must not break when scraping
- [ ] Parse an HTML string into a tree with `BeautifulSoup(html, "html.parser")`
- [ ] Extract text, attributes, and links with both `find` / `find_all` and `.select` / `.select_one`
- [ ] Turn an HTML table into a pandas DataFrame
- [ ] Check `robots.txt` with `urllib.robotparser` before fetching, and throttle + cache requests
- [ ] Query, filter, and flatten OpenAlex JSON into pandas, guarding null fields
- [ ] Guard a scraper against a missing element (no `AttributeError` on `None`)
- [ ] Say when to reach for a managed scraper like Firecrawl

## 🚀 Next step

That's the fast track! 🎉 For the full depth — crawling, schema-driven extraction, the OpenAlex graph — see **[`../16_webscraping/`](../16_webscraping/)**, and open the full course [`../00_onboarding/00_master_onboarding.ipynb`](../00_onboarding/00_master_onboarding.ipynb).

You've climbed the whole data-acquisition ladder: prefer an **API** (OpenAlex), fall back to **polite static-HTML** scraping (BeautifulSoup + `robots.txt`), and reach for a **managed scraper** (Firecrawl) only when JavaScript or anti-bot defences leave no choice. Most of the world's data has no API — now you can get it anyway, *politely*.